# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print main metadata
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.date_published}")
print(f"Version: {getattr(dataset.metadata, 'version', 'N/A')}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, their column `@id`s, and example records.


In [ ]:
# List all available record sets and their field/column @ids
record_sets = dataset.metadata.record_sets
if not record_sets:
    print('No record sets found.')
else:
    for record_set in record_sets:
        print(f"RecordSet: {record_set.name}")
        print(f"  @id: {record_set.id}")
        print(f"  Description: {getattr(record_set, 'description', '-')}")
        fields = getattr(record_set, 'fields', [])
        if fields:
            print("  Fields:")
            for field in fields:
                print(f"    - {field.name} (@id: {field.id}) -> column: {getattr(field, 'column', '-')}")
        print()
# Print example records for first record set
if record_sets:
    rs = record_sets[0]
    print(f"First 3 records from RecordSet '{rs.name}' (@id: {rs.id}):")
    for i, rec in enumerate(dataset.records(record_set=rs.id)):
        print(rec)
        if i >= 2:
            break

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. Use `@id` values for reference.


In [ ]:
# List all available record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print(f"Record set @ids: {record_set_ids}")

# Load records for all record sets
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded record set '{rs_id}' with shape {df.shape}")
# Preview the main record set (first one)
main_rs_id = record_set_ids[0] if record_set_ids else None
if main_rs_id:
    print(f"\nColumns in '{main_rs_id}':")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filtering records, normalizing numeric fields, grouping by categories, and other relevant analytics. All references to fields/columns use their `@id`s (use the actual column names if the DataFrame columns differ from `@id`).


In [ ]:
# Example: Let's identify a numeric field by its '@id' in the main record set
main_df = dataframes.get(main_rs_id)
if main_df is not None and not main_df.empty:
    # Attempt to auto-select the first numeric column available
    numeric_cols = main_df.select_dtypes(include='number').columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]  # Use this as an example
        print(f"Numeric field (by column name): {numeric_field_id}")
    else:
        # Fallback if no numeric columns exist
        numeric_field_id = main_df.columns[0]
        print("No numeric column found: using first column as example.")

    threshold = main_df[numeric_field_id].mean() if numeric_field_id in main_df else 0
    # Filter: values above the mean
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where '{numeric_field_id}' > {threshold:0.2f}: {len(filtered_df)} records")

    # Normalize
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Head of filtered and normalized:")
    display(filtered_df[[numeric_field_id, normalized_col]].head())

    # Attempt to group by a string/categorical field (excluding the numeric field)
    group_field = None
    for col in main_df.columns:
        if col != numeric_field_id and pd.api.types.is_object_dtype(main_df[col]):
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"Grouped mean of '{numeric_field_id}' by '{group_field}':")
        display(grouped_df.head())
    else:
        print("No suitable group field found for groupby operation.")
else:
    print("Unable to perform EDA: main record set not found or empty.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and not main_df.empty:
    # Histogram for the main numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in Main Record Set")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If a group_field was detected, show its effect on the numeric variable
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=main_df)
        plt.title(f"'{numeric_field_id}' by '{group_field}'")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("No data for visualization.")

## 6. Conclusion
This notebook provides an overview and EDA of the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using `mlcroissant`.

- The dataset contains record sets, fields, and columns described by FAIR principles.
- Programmatic access via Croissant enables dynamic schema and data exploration.
- We previewed data, filtered and normalized a numeric field, performed groupwise summaries, and visualized distributions.

Continue with in-depth analysis or machine learning tasks as appropriate for your research workflow.